# Actividad 1 - Individual
## Análisis de imagen con detalles ocultos: canales de color y operaciones punto a punto


| | |
|---|---|
| **Asignatura** | Análisis de Imágenes y Vídeo |
| **Descripción** | Análisis de una imagen oscura mediante canales de color, operaciones punto a punto y ecualización de histograma. |
| **Alumno** | José Luis Risco Velasco |
| **Grado** | Ciencia de datos |
| **Fecha** | 06/04/2026 |
---
**Contenidos:** 0. Librerías · 1. Carga · 2. Canales · 3. Operaciones punto a punto · 4. Ecualización · 5. Análisis de colores


## 0. Importación de librerías

Se importan las librerías principales de visión artificial y visualización.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, exposure, img_as_float, img_as_ubyte
from skimage.color import rgb2hsv

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

print('OpenCV version :', cv2.__version__)
print('Librerías cargadas correctamente ✓')


## 1. Carga de la imagen con OpenCV y scikit-image

Ambas librerías cargan la imagen como arrays NumPy, pero difieren en el **orden de canales**:

| Librería | Orden de canales | Nota |
|---|---|---|
| **OpenCV** | **BGR** (Blue-Green-Red) | Hay que convertir a RGB para visualizar con Matplotlib |
| **scikit-image** | **RGB** (Red-Green-Blue) | Estándar; compatible directamente con Matplotlib |


In [ ]:
IMG_PATH = 'data/Img1_ActividadIndividual.png'

# ── OpenCV: carga en BGR y conversión a RGB ──────────────────────────────
img_bgr     = cv2.imread(IMG_PATH)
img_cv2_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# ── scikit-image: carga directamente en RGB ──────────────────────────────
img_sk = io.imread(IMG_PATH)

print(f'Forma   (OpenCV)      : {img_bgr.shape}  →  (Alto, Ancho, Canales)')
print(f'Forma   (scikit-image): {img_sk.shape}')
print(f'Tipo de dato          : {img_sk.dtype}')
print(f'Rango de intensidades : [{img_sk.min()}, {img_sk.max()}]')
print(f'Media global          : {img_sk.mean():.2f} / 255  ({img_sk.mean()/255*100:.1f}% del brillo máximo)')

# Verificación de coherencia entre ambas librerías
assert np.array_equal(img_cv2_rgb, img_sk[:,:,:3])
print('\nAmbas librerías producen el mismo resultado ✓')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_cv2_rgb)
axes[0].set_title('OpenCV  (BGR → RGB)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(img_sk)
axes[1].set_title('scikit-image  (RGB nativo)', fontsize=12)
axes[1].axis('off')
plt.suptitle('Imagen original - muy oscura (solo 18% del brillo máximo)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Canales de color

### ¿Cuántos canales tiene la imagen? ¿Qué representa cada canal?

La imagen tiene **3 canales** correspondientes al modelo de color **RGB (Red-Green-Blue)**:

| Canal | Índice NumPy | Nombre | Descripción |
|---|---|---|---|
| **R** | `[:,:,0]` | *Red* - Rojo | Intensidad de luz **roja** en cada píxel [0-255] |
| **G** | `[:,:,1]` | *Green* - Verde | Intensidad de luz **verde** en cada píxel [0-255] |
| **B** | `[:,:,2]` | *Blue* - Azul | Intensidad de luz **azul** en cada píxel [0-255] |

Cada canal es una matriz 2D de enteros sin signo de 8 bits (`uint8`). La combinación aditiva de los tres produce el color percibido.

> 💡 La media global (~47/255 ≈ 18% del máximo) confirma la subexposición severa que oculta los detalles.


In [ ]:
# Extracción de los tres canales
R = img_sk[:, :, 0]   # Canal Rojo
G = img_sk[:, :, 1]   # Canal Verde
B = img_sk[:, :, 2]   # Canal Azul

print(f'Número de canales : {img_sk.shape[2]}')
print(f'Resolución        : {img_sk.shape[1]} × {img_sk.shape[0]} píxeles')
print()
print('Estadísticas por canal (min / media / max / desv.std):')
for nombre, canal in [('R (Rojo) ', R), ('G (Verde)', G), ('B (Azul) ', B)]:
    print(f'  {nombre}: {canal.min():3d} / {canal.mean():6.2f} / {canal.max():3d} / ±{canal.std():.2f}')

# Visualización de los canales con mapa de color propio
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(img_sk)
axes[0].set_title('Original (RGB)', fontsize=11)
axes[0].axis('off')
for ax, canal, nombre, cmap in zip(axes[1:],
        [R, G, B],
        ['Canal R - Rojo', 'Canal G - Verde', 'Canal B - Azul'],
        ['Reds', 'Greens', 'Blues']):
    im2 = ax.imshow(canal, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f'{nombre}\nMedia={canal.mean():.1f} | Max={canal.max()}', fontsize=10)
    ax.axis('off')
    plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle('Figura 1 - Descomposición en canales RGB', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Operaciones punto a punto

Las **operaciones punto a punto** (*pointwise operations*) transforman cada píxel de forma **independiente** mediante $s = f(r)$. Son la herramienta principal para corrección de brillo y contraste.

Se aplican dos tipos:

| Transformación | Fórmula | Parámetro elegido | Efecto |
|---|---|---|---|
| **Gamma (potencia)** | $s=(r/255)^{\gamma}\times255$ | **γ = 0.40** | Aclara sombras preservando estructura |
| **Logarítmica** | $s=c\cdot\log(1+r)$ | c auto | Expande oscuros, comprime brillos |


### 3.1 Corrección Gamma (transformación de potencia)

Con **γ < 1** la curva de transformación es cóncava hacia arriba: los valores oscuros se mapean a valores significativamente más altos, revelando los detalles ocultos. Se implementa mediante una **LUT** (*Look-Up Table*) de 256 valores para mayor eficiencia.

In [ ]:
def gamma_correction(img_rgb, gamma):
    """Corrección gamma mediante tabla de búsqueda (LUT) de 256 entradas.
    gamma < 1  →  aclara la imagen (expande el rango oscuro)
    gamma > 1  →  oscurece la imagen
    """
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    return lut[img_rgb]

# Comparación de tres valores de gamma
gamma_025 = gamma_correction(img_sk, 0.25)   # muy agresivo
gamma_040 = gamma_correction(img_sk, 0.40)   # óptimo elegido ✓
gamma_055 = gamma_correction(img_sk, 0.55)   # suave

print('Resultados de corrección gamma:')
for g, im in [(0.25, gamma_025), (0.40, gamma_040), (0.55, gamma_055)]:
    print(f'  γ={g}  →  media: {img_sk.mean():.1f}  →  {im.mean():.1f}')

imgs_g = [img_sk, gamma_025, gamma_040, gamma_055]
tits_g = ['Original', 'γ = 0.25\n(muy agresivo)', 'γ = 0.40\n(óptimo ✓)', 'γ = 0.55\n(suave)']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Figura 2 - Corrección Gamma: comparación de exponentes', fontsize=13, fontweight='bold')
for ax, im, t in zip(axes[0], imgs_g, tits_g):
    ax.imshow(im); ax.set_title(t, fontsize=11); ax.axis('off')
for ax, im in zip(axes[1], imgs_g):
    for i, c in enumerate(['red', 'green', 'blue']):
        cnt, _ = np.histogram(im[:,:,i], bins=256, range=(0,255))
        ax.plot(cnt, color=c, alpha=0.75, linewidth=0.9)
    ax.set_xlim(0,255); ax.set_xlabel('Intensidad', fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 3.2 Transformación Logarítmica

$$s = c \cdot \log(1 + r), \qquad c = \frac{255}{\log(1 + r_{\max})}$$

El logaritmo **expande los valores oscuros** y **comprime los brillos extremos**, siendo especialmente útil cuando la imagen tiene picos muy concentrados en la zona oscura.

In [ ]:
def log_transform(img_rgb):
    """Transformación logarítmica: s = c * log(1 + r).
    c se calcula automáticamente para que el máximo de salida sea 255.
    """
    img_f = img_rgb.astype(np.float64)
    c = 255.0 / np.log(1 + img_f.max())
    return np.clip(c * np.log(1 + img_f), 0, 255).astype(np.uint8)

img_log = log_transform(img_sk)

c_val = 255.0 / np.log(1 + img_sk.max())
print(f'Constante c calculada : {c_val:.4f}')
print(f'Media original        : {img_sk.mean():.2f}')
print(f'Media tras log        : {img_log.mean():.2f}  (+{img_log.mean()-img_sk.mean():.1f} puntos)')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figura 3 - Transformación Logarítmica', fontsize=13, fontweight='bold')
axes[0].imshow(img_sk); axes[0].set_title('Original', fontsize=12); axes[0].axis('off')
axes[1].imshow(img_log)
axes[1].set_title(r'Log:  $s = c \cdot \log(1+r)$', fontsize=12); axes[1].axis('off')
for i, c in enumerate(['red', 'green', 'blue']):
    cnt, _ = np.histogram(img_log[:,:,i], bins=256, range=(0,255))
    axes[2].plot(cnt, color=c, alpha=0.85, linewidth=1.0, label=['R','G','B'][i])
axes[2].set_title('Histograma post-Log', fontsize=12)
axes[2].set_xlabel('Intensidad'); axes[2].set_ylabel('Frecuencia')
axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 4. Ecualización del histograma

La **ecualización del histograma** redistribuye la función de densidad de probabilidad (PDF) de las intensidades para que la función de distribución acumulada (CDF) sea lo más lineal posible, maximizando el contraste global de la imagen.

Se implementan **3 métodos** distintos:

| # | Método | Herramienta | Ventaja |
|---|---|---|---|
| 1a | **HE clásica - canal a canal** | `cv2.equalizeHist()` sobre R, G, B | Simple y rápido |
| 1b | **HE sobre canal V en HSV** | `cv2.equalizeHist()` sobre canal de brillo | Preserva colores |
| 2  | **CLAHE en espacio LAB** | `cv2.createCLAHE()` sobre canal L | Adaptativo, sin artefactos |
| 3  | **HE adaptativo scikit-image** | `exposure.equalize_adapthist()` | API de alto nivel |


In [ ]:
# ── Método 1a: Ecualización global - canal a canal RGB ───────────────────
def equalize_rgb_classic(img):
    """cv2.equalizeHist() aplicado de forma independiente a cada canal R, G y B."""
    res = np.zeros_like(img)
    for i in range(3):
        res[:,:,i] = cv2.equalizeHist(img[:,:,i])
    return res

# ── Método 1b: HE sobre el canal V del espacio HSV ───────────────────────
def equalize_hsv(img):
    """Equaliza solo el brillo (canal V) en HSV.
    El matiz (H) y la saturación (S) no se modifican → preserva los colores.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hsv[:,:,2] = cv2.equalizeHist(hsv[:,:,2])
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

img_eq_classic = equalize_rgb_classic(img_sk)
img_eq_hsv     = equalize_hsv(img_sk)

print(f'Método 1a (HE RGB) - media resultado: {img_eq_classic.mean():.2f}')
print(f'Método 1b (HE HSV) - media resultado: {img_eq_hsv.mean():.2f}')


In [ ]:
# ── Método 2: CLAHE en espacio CIELab ───────────────────────────────────
def clahe_lab(img, clip_limit=3.0, tile_size=(8,8)):
    """CLAHE (Contrast Limited Adaptive Histogram Equalization) sobre canal L del LAB.
    clip_limit : límite máximo de amplificación (previene ruido excesivo)
    tile_size  : tamaño de las subregiones de análisis local
    """
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    lab[:,:,0] = clahe.apply(lab[:,:,0])   # Solo canal de luminancia L
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

# ── Método 3: HE adaptativo - scikit-image ────────────────────────────────
img_sk_float = img_as_float(img_sk)        # uint8 → float [0.0, 1.0]
img_eq_ski   = img_as_ubyte(
    exposure.equalize_adapthist(img_sk_float, clip_limit=0.03)
)                                           # float → uint8 [0, 255]

img_clahe = clahe_lab(img_sk, clip_limit=3.0)

print(f'Método 2 (CLAHE, clip=3)    - media: {img_clahe.mean():.2f}')
print(f'Método 3 (scikit-image)     - media: {img_eq_ski.mean():.2f}')


In [ ]:
# ── Figura 4: Comparación visual de todos los métodos ────────────────────
imgs_eq = [img_sk, img_eq_classic, img_eq_hsv, img_clahe, img_eq_ski]
tits_eq = [
    'Original',
    'Método 1a: HE clásica\n(canal a canal RGB)',
    'Método 1b: HE en HSV\n(canal V)',
    'Método 2: CLAHE\n(clip=3, espacio LAB)',
    'Método 3: Adapt. HE\n(scikit-image)',
]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle('Figura 4 - Comparación de métodos de ecualización del histograma',
             fontsize=13, fontweight='bold')

# Fila superior: imágenes resultantes
for ax, im, t in zip(axes[0], imgs_eq, tits_eq):
    ax.imshow(im); ax.set_title(t, fontsize=10); ax.axis('off')

# Fila inferior: histogramas RGB de cada resultado
for ax, im in zip(axes[1], imgs_eq):
    for i, c in enumerate(['red', 'green', 'blue']):
        cnt, _ = np.histogram(im[:,:,i], bins=256, range=(0,255))
        ax.plot(cnt, color=c, alpha=0.75, linewidth=0.8)
    ax.set_xlim(0,255); ax.set_xlabel('Intensidad', fontsize=8)
    ax.set_ylabel('Frecuencia', fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Figura 5: CDF - efecto de la ecualización
# Tras una ecualización perfecta, la CDF debería ser una diagonal (distribución uniforme)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figura 5 - Función de Distribución Acumulada (CDF) por método',
             fontsize=12, fontweight='bold')

pares = [
    (img_sk,     'Original (sin ecualizar)'),
    (img_clahe,  'Método 2: CLAHE (clip=3)'),
    (img_eq_ski, 'Método 3: Adapt. HE scikit-image'),
]

for ax, (im, t) in zip(axes, pares):
    for i, c in enumerate(['red', 'green', 'blue']):
        cnt, _ = np.histogram(im[:,:,i], bins=256, range=(0,255))
        cdf = cnt.cumsum() / cnt.sum()
        ax.plot(cdf, color=c, linewidth=1.8, label=['R','G','B'][i])
    # CDF ideal (distribución uniforme perfecta)
    ax.plot(np.linspace(0,1,256), 'k--', linewidth=1.2, alpha=0.6, label='Ideal')
    ax.set_title(t, fontsize=10)
    ax.set_xlabel('Intensidad'); ax.set_ylabel('CDF')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xlim(0,255)

plt.tight_layout()
plt.show()


## 5. Análisis y descripción de los colores de la imagen

Para identificar los colores presentes con precisión se analiza la imagen con **CLAHE** (que revela los colores reales sin artefactos) en el espacio **HSV**, separando el matiz (Hue) del brillo (Value).


In [ ]:
best = img_clahe   # imagen con mejor revelado de color

# Conversión a HSV (scikit-image devuelve valores en [0,1])
img_hsv_sk = rgb2hsv(best)
hue_deg    = img_hsv_sk[:,:,0] * 360   # matiz en grados [0°, 360°]
sat        = img_hsv_sk[:,:,1]          # saturación [0, 1]
val        = img_hsv_sk[:,:,2]          # valor/brillo [0, 1]

print('=== Estadísticas de color (imagen con CLAHE) ===')
print(f'Matiz - media: {hue_deg.mean():.1f}°  |  mediana: {np.median(hue_deg):.1f}°')
print(f'Saturación media : {sat.mean():.3f}  (rango 0-1)')
print(f'Brillo medio     : {val.mean():.3f}  (rango 0-1)')
print()

# Distribución de píxeles por rango de matiz (color)
rangos = [
    ('Rojo/Magenta',     (330, 360)),
    ('Rojo',             (  0,  20)),
    ('Naranja',          ( 20,  45)),
    ('Amarillo',         ( 45,  70)),
    ('Verde',            ( 70, 160)),
    ('Azul',             (200, 260)),
    ('Violeta',          (260, 330)),
]
total = hue_deg.size
print('Distribución de colores (% de píxeles):')
for nombre, (lo, hi) in rangos:
    mask = (hue_deg >= lo) & (hue_deg < hi)
    pct  = 100 * mask.sum() / total
    if pct > 1.0:
        print(f'  {nombre:20s}: {pct:5.1f}%')


In [ ]:
# Figura 6: Panel completo de análisis de color
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Figura 6 - Análisis de Color de la Imagen', fontsize=14, fontweight='bold')

# [0,0] Imagen con CLAHE
axes[0,0].imshow(best)
axes[0,0].set_title('Imagen con CLAHE\n(colores revelados)', fontsize=11)
axes[0,0].axis('off')

# [0,1] Histograma de Matiz (Hue)
axes[0,1].hist(hue_deg.flatten(), bins=180, range=(0,360), color='purple', alpha=0.7)
axes[0,1].axvspan( 0,  45, alpha=0.18, color='orange', label='Rojo/Naranja')
axes[0,1].axvspan(45,  70, alpha=0.18, color='yellow', label='Amarillo')
axes[0,1].axvspan(70, 160, alpha=0.12, color='green',  label='Verde')
axes[0,1].axvspan(200,270, alpha=0.12, color='blue',   label='Azul')
axes[0,1].set_title('Histograma de Matiz (Hue)\nEspacio HSV', fontsize=11)
axes[0,1].set_xlabel('Matiz (°)'); axes[0,1].set_ylabel('Nº de píxeles')
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# [0,2] Saturación y Valor
axes[0,2].hist(sat.flatten(), bins=100, color='teal', alpha=0.7, label='Saturación')
axes[0,2].hist(val.flatten(), bins=100, color='gold', alpha=0.5, label='Valor (brillo)')
axes[0,2].set_title('Saturación y Valor\nEspacio HSV', fontsize=11)
axes[0,2].set_xlabel('Valor normalizado [0,1]')
axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

# [1, 0-2] Canales R, G, B de la imagen CLAHE
for i, (ax, nombre, cmap) in enumerate(zip(
        axes[1],
        ['R - Rojo', 'G - Verde', 'B - Azul'],
        ['Reds', 'Greens', 'Blues'])):
    im2 = ax.imshow(best[:,:,i], cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f'Canal {nombre} (CLAHE)\nMedia={best[:,:,i].mean():.1f}', fontsize=10)
    ax.axis('off')
    plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


### ✅ Descripción de los colores de la imagen

Tras revelar la imagen mediante corrección gamma (γ=0.40), transformación logarítmica y ecualización CLAHE, se identifican con claridad los siguientes colores:

| Color | % aprox. | Origen en la imagen |
|---|---|---|
| 🟠 **Naranja / Dorado** | ~29% | Luz cálida del atardecer iluminando el fondo y el contorno de la figura |
| 🔴 **Rojo / Rojizo** | ~16% | Reflejos solares en el cabello y en el cielo del horizonte |
| 🟡 **Amarillo** | ~18% | Campo de flores y hierbas iluminadas en el fondo |
| 🟢 **Verde amarillento** | ~15% | Vegetación: hierbas y árbol al fondo de la escena |
| ⚫ **Negro / Muy oscuro** | ~15% | Zonas de sombra profunda: rostro, torso y pelo interior |
| 🔵 **Azul grisáceo** | ~4% | Vestido de la persona |
| ⚪ **Blanco / Crema** | <2% | Corona de flores sobre la cabeza |

**Conclusión:** La imagen presenta una paleta **cálida y de alto contraste**, dominada por los tonos **naranja-dorado, rojo y amarillo** propios de un contraluz de atardecer. El matiz mediano (~25°, rojo-naranja) y la saturación moderada (≈ 0.41) son coherentes con una iluminación de hora dorada: la luz directa es muy cálida pero las zonas en sombra son muy oscuras. Los únicos tonos fríos presentes son el **azul grisáceo del vestido** y el **blanco de la corona floral**, que quedan ocultos en la imagen original debido a la severa subexposición.
